# 🦆 Laboratório: Processamento de Dados com DuckDB e Pandas

> **Objetivo:** Aprender a usar o DuckDB como motor de banco de dados embutido no Python para manipular grandes volumes de dados de forma eficiente — mesmo em máquinas com recursos limitados (ex: 8 GB de RAM).

---

## 📋 Pré-requisitos

| Requisito | Versão Mínima |
|-----------|---------------|
| Python    | 3.9+          |
| DuckDB    | 0.10+         |
| Pandas    | 1.5+          |
| NumPy     | 1.23+         |

## 🗺️ Mapa do Laboratório

1. [Configuração do Ambiente](#1-configuração-do-ambiente)
2. [DuckDB como Banco de Dados Persistente](#2-duckdb-como-banco-de-dados-persistente)
3. [Integração de Alta Performance com Pandas](#3-integração-de-alta-performance-com-pandas)
4. [Processando Arquivos Grandes com Pouca RAM](#4-processando-arquivos-grandes-com-pouca-ram)
5. [Desafio Prático](#5-desafio-prático)
6. [Referência de Comandos](#6-referência-de-comandos)


---

## 1. Configuração do Ambiente

Instale as bibliotecas necessárias e verifique as versões. Execute a célula abaixo — se tudo carregar sem erros, o ambiente está pronto.

In [ ]:
# Descomente a linha abaixo caso não tenha as bibliotecas instaladas
# !pip install duckdb pandas numpy

import duckdb
import pandas as pd
import numpy as np

print(f"✅ DuckDB  : {duckdb.__version__}")
print(f"✅ Pandas  : {pd.__version__}")
print(f"✅ NumPy   : {np.__version__}")
print("\nAmbiente configurado com sucesso!")

---

## 2. DuckDB como Banco de Dados Persistente

### 💡 Conceito

O DuckDB é o **substituto moderno do SQLite para análise de dados**. Diferente de servidores como PostgreSQL ou MySQL, ele roda dentro do seu processo Python e guarda tudo em um único arquivo local — sem instalação de servidor, sem configuração de rede.

```
┌──────────────┐       SQL        ┌─────────────────────┐
│  Python App  │ ◄─────────────► │  DuckDB (embutido)  │
│   / Jupyter  │                  │   meubanco.db 💾    │
└──────────────┘                  └─────────────────────┘
```

> **Quando usar banco persistente?** Quando você quer reutilizar os dados entre sessões do Jupyter sem reprocessar tudo.

In [ ]:
# Criar ou conectar a um banco de dados no disco
# Se o arquivo não existir, o DuckDB cria automaticamente
con = duckdb.connect("meubanco.db")

print("✅ Conectado ao banco 'meubanco.db'")

In [ ]:
# Criar a tabela (IF NOT EXISTS evita erros ao rodar novamente)
con.sql("""
    CREATE TABLE IF NOT EXISTS usuarios (
        id   INTEGER PRIMARY KEY,
        nome TEXT NOT NULL
    )
""")

# Limpar dados antigos para evitar duplicatas ao reexecutar
con.sql("DELETE FROM usuarios")

# Inserir registros
con.sql("INSERT INTO usuarios VALUES (1, 'Lira'), (2, 'Alexandre'), (3, 'Eiko')")

print("✅ Tabela criada e dados inseridos")

In [ ]:
# Consultar os dados
print("📋 Conteúdo da tabela 'usuarios':")
con.sql("SELECT * FROM usuarios ORDER BY id").show()

# IMPORTANTE: sempre feche a conexão ao terminar
# Isso garante que o arquivo .db seja liberado corretamente
con.close()
print("\n🔒 Conexão encerrada")

> **⚠️ Boas práticas:** Sempre chame `con.close()` ao terminar. Alternativamente, use o gerenciador de contexto:
> ```python
> with duckdb.connect("meubanco.db") as con:
>     con.sql("SELECT * FROM usuarios").show()
> # A conexão é fechada automaticamente aqui
> ```

---

## 3. Integração de Alta Performance com Pandas

### 💡 Conceito

O DuckDB consegue **ler DataFrames Pandas diretamente da memória** sem cópia de dados — ele enxerga a variável Python como se fosse uma tabela SQL. Isso elimina o overhead de importação e torna as consultas extremamente rápidas.

```
   DataFrame Pandas          DuckDB                Resultado
  ┌───────────────┐    ┌──────────────┐       ┌───────────────┐
  │  df_vendas    │───►│  SELECT ...  │──────►│  df_resultado │
  │  (na memória) │    │  FROM df_... │  .df()│  (Pandas)     │
  └───────────────┘    └──────────────┘       └───────────────┘
```

In [ ]:
# Criar um DataFrame de vendas de exemplo
df_vendas = pd.DataFrame({
    'data'     : pd.to_datetime(['2023-09-25', '2023-10-01', '2023-10-01', '2023-10-15', '2023-10-20']),
    'produto'  : ['Cadeira', 'Monitor', 'Teclado', 'Webcam', 'Mesa'],
    'categoria': ['Móveis', 'Eletrônicos', 'Acessórios', 'Eletrônicos', 'Móveis'],
    'valor'    : [770, 1479, 200, 350, 1200],
    'qtd'      : [10, 3, 7, 5, 2],
    'status'   : ['Ativo', 'Cancelado', 'Ativo', 'Ativo', 'Ativo']
})

print("📦 Dataset de vendas:")
display(df_vendas)

In [ ]:
# Executar SQL diretamente sobre a variável 'df_vendas'
# O DuckDB reconhece automaticamente o DataFrame pelo nome da variável
query = """
    SELECT 
        categoria,
        COUNT(*)          AS num_produtos,
        AVG(valor)        AS ticket_medio,
        SUM(qtd)          AS total_unidades,
        SUM(valor * qtd)  AS faturamento_total
    FROM df_vendas
    WHERE status = 'Ativo'
    GROUP BY categoria
    ORDER BY faturamento_total DESC
"""

# .df() converte o resultado de volta para um DataFrame Pandas
df_resultado = duckdb.sql(query).df()

print("📊 Análise por categoria (somente pedidos Ativos):")
display(df_resultado)

### Conversões disponíveis

| Método | Retorna |
|--------|---------|
| `.df()` | `pandas.DataFrame` |
| `.pl()` | `polars.DataFrame` (requer `pip install polars`) |
| `.arrow()` | `pyarrow.Table` |
| `.fetchall()` | `list` de tuplas Python |
| `.show()` | Exibe no terminal (sem retorno) |

---

## 4. Processando Arquivos Grandes com Pouca RAM

### 💡 O Problema

Imagine um arquivo CSV com 50 milhões de linhas e 5 GB. O Pandas carrega **tudo na RAM** antes de filtrar:

```python
# ❌ Abordagem Pandas: ocupa 5 GB de RAM imediatamente
df = pd.read_csv("gigante.csv")       # RAM: 5 GB
df_filtrado = df[df['status'] == 'Ativo']  # Cópia na memória
```

### ✅ A Solução com DuckDB

O DuckDB lê o arquivo em **streaming** — processa pedaço por pedaço sem carregar tudo na memória:

```
arquivo.csv (5 GB)          DuckDB                 Resultado
┌─────────────────┐   ┌────────────────┐       ┌──────────────┐
│ chunk 1 (256 MB)│──►│                │       │              │
│ chunk 2 (256 MB)│──►│  WHERE + GROUP │──────►│  ~10 linhas  │
│ chunk 3 (256 MB)│──►│      BY        │  .df()│  (kB na RAM) │
│       ...       │   │                │       │              │
└─────────────────┘   └────────────────┘       └──────────────┘
   Disco (SSD)          Máx ~256 MB RAM          Seu DataFrame
```

In [ ]:
# Vamos simular um arquivo CSV grande para demonstração
import tempfile, os

NUM_LINHAS = 500_000  # Simula 500 mil registros

rng = np.random.default_rng(42)
df_simulado = pd.DataFrame({
    'produto'  : rng.choice(['Cadeira', 'Monitor', 'Teclado', 'Webcam', 'Mesa'], NUM_LINHAS),
    'categoria': rng.choice(['Móveis', 'Eletrônicos', 'Acessórios'], NUM_LINHAS),
    'valor'    : rng.integers(50, 5000, NUM_LINHAS),
    'qtd'      : rng.integers(1, 50, NUM_LINHAS),
    'status'   : rng.choice(['Ativo', 'Cancelado', 'Pendente'], NUM_LINHAS, p=[0.7, 0.2, 0.1]),
})

csv_path = "/tmp/vendas_simuladas.csv"
df_simulado.to_csv(csv_path, index=False)

tamanho_mb = os.path.getsize(csv_path) / 1024 / 1024
print(f"✅ Arquivo gerado: {csv_path}")
print(f"   📏 Linhas   : {NUM_LINHAS:,}")
print(f"   💾 Tamanho  : {tamanho_mb:.1f} MB")

In [ ]:
import time

# --- Abordagem DuckDB (streaming do CSV) ---
t0 = time.perf_counter()

df_agregado = duckdb.sql(f"""
    SELECT 
        produto,
        categoria,
        COUNT(*)          AS num_vendas,
        SUM(valor * qtd)  AS faturamento_total
    FROM '{csv_path}'
    WHERE status = 'Ativo'
    GROUP BY produto, categoria
    ORDER BY faturamento_total DESC
""").df()

tempo_duckdb = time.perf_counter() - t0

print(f"⏱️  DuckDB (streaming): {tempo_duckdb:.3f}s")
print(f"📊 Resultado ({len(df_agregado)} linhas):")
display(df_agregado)

In [ ]:
# --- Abordagem Pandas (para comparação) ---
t0 = time.perf_counter()

df_pd = pd.read_csv(csv_path)  # Carrega tudo na RAM
df_pd_resultado = (
    df_pd[df_pd['status'] == 'Ativo']
    .assign(faturamento_total=lambda x: x['valor'] * x['qtd'])
    .groupby(['produto', 'categoria'])
    .agg(num_vendas=('valor', 'count'), faturamento_total=('faturamento_total', 'sum'))
    .reset_index()
    .sort_values('faturamento_total', ascending=False)
)

tempo_pandas = time.perf_counter() - t0

print(f"⏱️  Pandas (read_csv): {tempo_pandas:.3f}s")
print(f"\n🏆 DuckDB foi {tempo_pandas/tempo_duckdb:.1f}x mais rápido neste exemplo")

### Salvar resultado em Parquet (formato colunar eficiente)

Parquet é o formato ideal para armazenar dados processados: compressão excelente, leitura rápida por coluna, e compatível com Spark, BigQuery, Athena e afins.

In [ ]:
# Salvar o resultado processado direto em Parquet — sem passar pelo Pandas
duckdb.sql(f"""
    COPY (
        SELECT produto, categoria, SUM(valor * qtd) AS faturamento_total
        FROM '{csv_path}'
        WHERE status = 'Ativo'
        GROUP BY produto, categoria
    ) TO '/tmp/resultado.parquet' (FORMAT PARQUET)
""")

tamanho_parquet = os.path.getsize('/tmp/resultado.parquet') / 1024
print(f"✅ Salvo em '/tmp/resultado.parquet' ({tamanho_parquet:.1f} KB)")

# Ler de volta para verificar
df_check = duckdb.sql("SELECT * FROM '/tmp/resultado.parquet'").df()
display(df_check)

---

## 5. Desafio Prático

### 🎯 Enunciado

Usando o `df_vendas` criado na Seção 3, escreva **uma única query SQL** que retorne:

1. O **faturamento total** (`valor * qtd`) por produto
2. Apenas produtos com `status = 'Ativo'`
3. Apenas da categoria `'Eletrônicos'`
4. Ordenado do maior para o menor faturamento

**Resultado esperado:**

| produto | faturamento_total |
|---------|-------------------|
| Webcam  | 1750              |

---

### ✏️ Sua solução:

In [ ]:
# ✏️ Escreva sua query aqui
minha_query = """
    -- Substitua pelo seu código SQL
    SELECT 'complete o desafio!' AS mensagem
"""

df_desafio = duckdb.sql(minha_query).df()
display(df_desafio)

In [ ]:
# 💡 GABARITO — tente resolver antes de olhar!
# (Execute esta célula para ver a solução)

gabarito = """
    SELECT 
        produto,
        SUM(valor * qtd) AS faturamento_total
    FROM df_vendas
    WHERE status    = 'Ativo'
      AND categoria = 'Eletrônicos'
    GROUP BY produto
    ORDER BY faturamento_total DESC
"""

display(duckdb.sql(gabarito).df())

---

## 6. Referência de Comandos

### Conexões

```python
duckdb.connect()           # Banco em memória (temporário)
duckdb.connect("file.db")  # Banco persistente em arquivo
```

### Execução de Queries

```python
duckdb.sql("SELECT ...")   # Conexão padrão (global)
con.sql("SELECT ...")      # Conexão específica
```

### Saída de Resultados

| Método | Descrição |
|--------|-----------|
| `.df()` | Converte para `pandas.DataFrame` |
| `.pl()` | Converte para `polars.DataFrame` |
| `.arrow()` | Converte para `pyarrow.Table` |
| `.fetchall()` | Retorna lista de tuplas Python |
| `.fetchone()` | Retorna a primeira linha como tupla |
| `.show()` | Exibe formatado no terminal |

### Leitura Direta de Arquivos

```python
# CSV
duckdb.sql("SELECT * FROM 'dados.csv'")

# Parquet
duckdb.sql("SELECT * FROM 'dados.parquet'")

# Múltiplos arquivos com glob
duckdb.sql("SELECT * FROM 'pasta/*.parquet'")
```

### Exportação

```python
# Parquet (recomendado)
duckdb.sql("COPY (SELECT ...) TO 'saida.parquet' (FORMAT PARQUET)")

# CSV
duckdb.sql("COPY (SELECT ...) TO 'saida.csv' (HEADER, DELIMITER ',')")
```

---

## 📚 Próximos Passos

- 📖 [Documentação oficial do DuckDB](https://duckdb.org/docs/)
- 🔌 [Extensão HTTPFS — ler arquivos direto de S3/GCS](https://duckdb.org/docs/extensions/httpfs)
- ⚡ [DuckDB + Polars — a combinação mais rápida](https://duckdb.org/docs/guides/python/polars)
- 🌐 [MotherDuck — DuckDB na nuvem](https://motherduck.com/)
